In [ ]:
from argparse import Namespace
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
import glob
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../resources/library/tropical_cyclone')
import dynamicopy
from tropical_cyclone.cyclone import (
    BYTETracker,
    clean_tracks,
)

In [ ]:
# define inference directory to draw detections
dataset_dir = f'path-to/data/jra-inference-folder-containing-csv-files'
available_models = sorted([folder for folder in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, folder))])
len(available_models)

In [ ]:
# select the model to analyze
selected_model_NHS = 'localization-classification-NORTH-HEMISPHERE-MODEL'
selected_model_SHS = 'localization-classification-SOUTH-HEMISPHERE-MODEL'
tracks_model = 'name-of-trained-bytestorm-model'

In [ ]:
# tracker arguments
bbox_size = 21      # 15, 21, 25, 31, 35 # default : 25
track_thresh = 0.7  # Soglia di confidenza per considerare una detection valida
track_buffer = 3    # Numero di frame per cui mantenere una traccia "persa"
match_thresh = 0.8  # Soglia di IoU per l'associazione tra tracce e nuove detection (più è alta più lontano guardiamo)
tracking_dir = os.path.join(dataset_dir, tracks_model)
tracking_src = os.path.join(tracking_dir, f'tracking_TT{track_thresh}-TB{track_buffer}-MT{match_thresh}-BB{bbox_size}-RAW.csv')
tracking_dst = os.path.join(tracking_dir, f'tracking_TT{track_thresh}-TB{track_buffer}-MT{match_thresh}-BB{bbox_size}-CLEAN.csv')
os.makedirs(tracking_dir, exist_ok=True)
tracking_src

In [ ]:
# define test years
test_years = [2020,2021,2022]

# convert lat and lon to row col (considering map as a matrix)
lat_lim = [-70, 70.25]
lon_lim = [0, 360]
lats, lons = np.arange(*lat_lim, 0.25)[::-1], np.arange(*lon_lim, 0.25)
img_size = (len(lats), len(lons))
img_size

In [ ]:
# get model directory
model_dir_NHS = os.path.join(dataset_dir, selected_model_NHS)
inference_files_NHS = [os.path.join(model_dir_NHS, f'{year}.csv') for year in test_years]
model_dir_NHS, len(inference_files_NHS)

In [ ]:
# get model directory
model_dir_SHS = os.path.join(dataset_dir, selected_model_SHS)
inference_files_SHS = [os.path.join(model_dir_SHS, f'{year}.csv') for year in test_years]
model_dir_SHS, len(inference_files_SHS)

In [ ]:
inference_files_NHS = [f'{dataset_dir}/{selected_model_NHS}/detections.csv']
inference_files_SHS = [f'{dataset_dir}/{selected_model_SHS}/detections.csv']

In [ ]:
# load csv files
dataframes = []
for file in inference_files_NHS + inference_files_SHS:
    dataframes.append(pd.read_csv(file, index_col=0))
# merge csv files together
detections = pd.concat(dataframes)
# convert iso time with pandas
detections['ISO_TIME'] = pd.to_datetime(detections['ISO_TIME'])
# sort by ISO_TIME
detections = detections.sort_values(by='ISO_TIME')
# reset index
detections = detections.reset_index(drop=True)
# add WS as np.inf
detections['WS'] = np.inf
detections.head()

In [ ]:
# convert lat and lon to row col (considering map as a matrix)
detections['YLAT'] = [np.argwhere(lats==l)[0][0] for l in detections['LAT']]
detections['XLON'] = [np.argwhere(lons==l)[0][0] for l in detections['LON']]
detections['BBOX'] = np.matrix.tolist(np.stack([detections['XLON'] - bbox_size / 2, detections['YLAT'] - bbox_size / 2, detections['XLON'] + bbox_size / 2, detections['YLAT'] + bbox_size / 2], axis=-1))

In [ ]:
if not os.path.exists(tracking_src):
    tracker = BYTETracker(
        track_thresh=track_thresh, 
        track_buffer=track_buffer, 
        match_thresh=match_thresh, 
        frame_rate=1, ratio=1, lats=lats, lons=lons
    )
    size = img_size
    grouped_detections = detections.groupby('ISO_TIME')
    # for each iso time in the dataset
    for iso_time, dets in tqdm(grouped_detections):
        output_result = torch.as_tensor(np.c_[np.stack(dets['BBOX']), np.stack(dets['PROB'])])
        tracker.update(output_result, img_info=size, img_size=size, date=iso_time)
    tracks = tracker.create_tracks_dataframe()
    tracks.to_csv(tracking_src)

In [ ]:
bytestorm = pd.read_csv(tracking_src, index_col=0)
print(len(bytestorm))
bytestorm = clean_tracks(dynamicopy, bytestorm, min_track_count=12)
print(len(bytestorm))
bytestorm.to_csv(tracking_dst)